##Data Entry

###Method:
1. Load in raw csv file for weather and for the ISO/RTO, input if CAISO or NYISO, input final file destination

2. Check if there is missing data

3. Impute using the methods from the CAISO imputation for CAISO and NYISO for NYISO

4. Compile the demand and weather into a wide table

5. Output the file (into the correct destination)

6. Output a visual of the weather and power

In [ ]:
from pathlib import Path
from typing import Optional, Sequence

import matplotlib.pyplot as plt
import pandas as pd


# =========================
# Core Functions
# =========================

def _ensure_datetime(df: pd.DataFrame, col: str):
    print(f"[datetime] Converting column '{col}' to datetime.")
    df = df.copy()
    df[col] = pd.to_datetime(df[col], errors="coerce")
    num_null = df[col].isna().sum()
    if num_null:
        print(f"[datetime] Column '{col}' has {num_null} unparseable timestamps converted to NaT.")
    return df


def find_first_existing(df: pd.DataFrame, candidates: Sequence[str]):
    for c in candidates:
        if c in df.columns:
            print(f"[schema] Found column '{c}' from candidates {list(candidates)}.")
            return c
    print(f"[schema] None of the candidate columns were found: {list(candidates)}.")
    return None


def normalize_nyiso_raw(df: pd.DataFrame):
    """
    This function converts the raw NYISO daily load schema to pipeline schema.
    * An important for imputation part of this function is that it keeps rows where 'Hourly Avg Load' is missing so they can be imputed later.
    """
    print("[nyiso] Normalizing raw NYISO load schema.")
    df = df.copy()
    starting_rows = len(df)

    df = df.rename(columns={
        "Time Stamp": "Hour Start",
        "Load": "Hourly Avg Load",
    })
    print("[nyiso] Renamed columns: 'Time Stamp' -> 'Hour Start', 'Load' -> 'Hourly Avg Load' where present.")

    df["Hour Start"] = pd.to_datetime(df["Hour Start"], errors="coerce")
    bad_ts = df["Hour Start"].isna().sum()
    if bad_ts:
        print(f"[nyiso] Found {bad_ts} rows with invalid Hour Start values; they will be dropped.")

    before_drop = len(df)
    df = df.dropna(subset=["Hour Start", "Name"]).copy()
    dropped = before_drop - len(df)
    print(f"[nyiso] Dropped {dropped} rows missing Hour Start or Name.")

    if "Imputed" not in df.columns:
        df["Imputed"] = "NO"
        print("[nyiso] Added missing 'Imputed' column with default value 'NO'.")
    else:
        df["Imputed"] = df["Imputed"].fillna("NO").astype(str)
        print("[nyiso] Filled null values in existing 'Imputed' column with 'NO'.")

    keep_cols = [
        c for c in [
            "Hour Start",
            "Name",
            "Hourly Avg Load",
            "PTID",
            "Time Zone",
            "Imputed",
        ]
        if c in df.columns
    ]

    print(f"[nyiso] Kept columns: {keep_cols}")
    print(f"[nyiso] NYISO normalization complete: {starting_rows} -> {len(df)} rows.")
    return df[keep_cols].copy()


def _standardize_load_columns(df: pd.DataFrame, iso: str):
    """
    This function standardizes load column names so the pipeline runs smoothly

    Output columns include at least:
      - Hour Start
      - Name
      - Hourly Avg Load
      - Imputed
    """
    iso = iso.upper()
    print(f"[load] Standardizing load columns for ISO={iso}.")
    df = df.copy()

    if iso == "NYISO":
        if "Time Stamp" in df.columns:
            print("[load] Detected raw NYISO schema with 'Time Stamp'. Running normalize_nyiso_raw().")
            df = normalize_nyiso_raw(df)

        if "Hour Start" not in df.columns:
            raise ValueError("NYISO load file must contain 'Hour Start' or 'Time Stamp'.")

        df["Hour Start"] = pd.to_datetime(df["Hour Start"], errors="coerce")
        before_drop = len(df)
        df = df.dropna(subset=["Hour Start"]).copy()
        print(f"[load] Dropped {before_drop - len(df)} NYISO rows with invalid Hour Start.")

        if "Name" not in df.columns:
            raise ValueError("NYISO load file must contain 'Name'.")

        load_col = find_first_existing(df, ["Hourly Avg Load", "Load", "load"])
        if load_col is None:
            raise ValueError("NYISO load file must contain a load column like 'Hourly Avg Load'.")

        if load_col != "Hourly Avg Load":
            df = df.rename(columns={load_col: "Hourly Avg Load"})
            print(f"[load] Renamed '{load_col}' to 'Hourly Avg Load'.")

        if "Imputed" not in df.columns:
            if "Imputed?" in df.columns:
                df = df.rename(columns={"Imputed?": "Imputed"})
                print("[load] Renamed 'Imputed?' to 'Imputed'.")
            else:
                df["Imputed"] = "NO"
                print("[load] Added missing 'Imputed' column with default value 'NO'.")
        else:
            df["Imputed"] = df["Imputed"].fillna("NO").astype(str)
            print("[load] Filled null values in 'Imputed' with 'NO'.")

        print(f"[load] NYISO load standardization complete. Rows: {len(df)}.")
        return df

    if iso == "CAISO":
        if "Hour Start" not in df.columns:
            if "Date" not in df.columns or "HR" not in df.columns:
                raise ValueError(
                    "CAISO load file must contain 'Hour Start' or both 'Date' and 'HR'."
                )

            print("[load] Building CAISO 'Hour Start' from 'Date' + 'HR'.")
            df = _ensure_datetime(df, "Date")
            before_drop = len(df)
            df = df[df["Date"].notna()].copy()
            print(f"[load] Dropped {before_drop - len(df)} CAISO rows with invalid Date.")
            df["Hour Start"] = df["Date"] + pd.to_timedelta(df["HR"], unit="h")

        if "CAISO" in df.columns and "CAISO Total" not in df.columns:
            df = df.rename(columns={"CAISO": "CAISO Total"})
            print("[load] Renamed 'CAISO' to 'CAISO Total'.")

        if "source_file" not in df.columns:
            df["source_file"] = "input_file"
            print("[load] Added default 'source_file' column.")

        value_cols = [c for c in ["PGE", "SCE", "SDGE", "VEA", "CAISO Total"] if c in df.columns]
        if not value_cols:
            raise ValueError(
                "CAISO load file needs at least one of: PGE, SCE, SDGE, VEA, CAISO Total."
            )

        print(f"[load] Melting CAISO utility columns into long format: {value_cols}")
        out = df.melt(
            id_vars=[c for c in ["Hour Start", "source_file"] if c in df.columns],
            value_vars=value_cols,
            var_name="Name",
            value_name="Hourly Avg Load",
        )
        out["Imputed"] = "NO"

        # IMPORTANT: do not drop missing Hourly Avg Load here.
        # Keep them so the same imputation logic used by NYISO can fill them.
        out = _ensure_datetime(out, "Hour Start")
        before_drop = len(out)
        out = out.dropna(subset=["Hour Start", "Name"]).reset_index(drop=True)
        print(f"[load] Dropped {before_drop - len(out)} CAISO rows with invalid Hour Start or Name.")

        print(f"[load] CAISO load standardization complete. Rows: {len(out)}.")
        return out

    raise ValueError("iso must be either 'CAISO' or 'NYISO'.") # how the pipeline is set up for now


# =========================
# Checking for missing datA
# =========================

def summarize_missing_data(load_df: pd.DataFrame, weather_df: pd.DataFrame):
    print("[check] Summarizing missing data and duplicates.")
    load_missing = load_df.isna().sum().sort_values(ascending=False)
    weather_missing = weather_df.isna().sum().sort_values(ascending=False)

    out = {
        "load_missing_counts": load_missing[load_missing > 0],
        "weather_missing_counts": weather_missing[weather_missing > 0],
        "load_has_missing": bool(load_missing.sum() > 0),
        "weather_has_missing": bool(weather_missing.sum() > 0),
    }

    if "Hour Start" in load_df.columns and "Name" in load_df.columns:
        tmp = load_df.copy()
        tmp["Hour Start"] = pd.to_datetime(tmp["Hour Start"], errors="coerce")
        out["load_duplicate_name_hour_rows"] = int(
            tmp.duplicated(subset=["Name", "Hour Start"]).sum()
        )

    if "Hour Start" in weather_df.columns:
        tmp = weather_df.copy()
        tmp["Hour Start"] = pd.to_datetime(tmp["Hour Start"], errors="coerce")
        zone_col = "Name" if "Name" in tmp.columns else "Zone" if "Zone" in tmp.columns else None
        if zone_col:
            out["weather_duplicate_name_hour_rows"] = int(
                tmp.duplicated(subset=[zone_col, "Hour Start"]).sum()
            )

    print("[check] Missing-data summary complete.")
    return out


# =========================
# Imptutation
# =========================

def impute_by_prior_week(load_df: pd.DataFrame, iso: str):
    """
    This function imputes missing hourly load by copying values from exactly X days earlier for the same Name and hour.

    This is the shared method for both NYISO and CAISO. It marks edited values as imputed.

    """
    iso = iso.upper()
    print(f"[impute][{iso}] Starting shared prior-time imputation.")

    required_cols = {"Name", "Hour Start", "Hourly Avg Load"}
    df = load_df.copy()

    if not required_cols.issubset(df.columns):
        raise ValueError(f"Input DataFrame must contain columns: {required_cols}")

    original_rows = len(df)

    df["Hour Start"] = pd.to_datetime(df["Hour Start"], errors="coerce")
    before_drop = len(df)
    df = df.dropna(subset=["Hour Start", "Name"]).copy()
    print(f"[impute][{iso}] Dropped {before_drop - len(df)} rows missing Hour Start or Name before imputation.")

    if "Imputed" not in df.columns:
        df["Imputed"] = "NO"
        print(f"[impute][{iso}] Added 'Imputed' column with default 'NO'.")
    else:
        df["Imputed"] = df["Imputed"].fillna("NO").astype(str)

    df = df.sort_values(["Name", "Hour Start"]).reset_index(drop=True)

    all_parts = []
    total_rows_added = 0
    total_bad_rows_replaced = 0

    for name, group in df.groupby("Name", sort=False):
        group = group.copy().sort_values("Hour Start").reset_index(drop=True)

        if group.empty:
            all_parts.append(group)
            continue

        min_ts = group["Hour Start"].min()
        max_ts = group["Hour Start"].max()
        full_hours = pd.date_range(start=min_ts, end=max_ts, freq="h")

        print(
            f"[impute][{iso}] Processing zone '{name}' "
            f"from {min_ts} to {max_ts} "
            f"({len(group)} observed rows, {len(full_hours)} expected hourly timestamps)."
        )

        group_for_lookup = group.copy()
        group_for_lookup["_load_is_null"] = group_for_lookup["Hourly Avg Load"].isna()
        group_for_lookup["_imputed_rank"] = group_for_lookup["Imputed"].map({"NO": 0, "YES": 1}).fillna(0)

        lookup = (
            group_for_lookup
            .sort_values(["Hour Start", "_load_is_null", "_imputed_rank"])
            .drop_duplicates(subset=["Hour Start"], keep="first")
            .set_index("Hour Start")
        )

        rows_to_add = []
        bad_existing_indices = []
        missing_count = 0
        filled_count = 0
        skipped_count = 0

        for ts in full_hours:
            needs_impute = False

            if ts not in lookup.index:
                needs_impute = True
            else:
                val = lookup.loc[ts, "Hourly Avg Load"]
                if pd.isna(val):
                    needs_impute = True

            if not needs_impute:
                continue

            missing_count += 1
            source_ts = ts - pd.Timedelta(days=7)

            if source_ts < min_ts:
                skipped_count += 1
                continue
            if source_ts not in lookup.index:
                skipped_count += 1
                continue

            source_row = lookup.loc[source_ts]
            if pd.isna(source_row["Hourly Avg Load"]):
                skipped_count += 1
                continue

            new_row = source_row.copy()
            new_row["Hour Start"] = ts
            new_row["Name"] = name
            new_row["Imputed"] = "YES"
            rows_to_add.append(new_row)
            filled_count += 1

            existing_bad = group[
                (group["Hour Start"] == ts) &
                (group["Hourly Avg Load"].isna())
            ].index.tolist()
            bad_existing_indices.extend(existing_bad)

        group_clean = group.drop(index=bad_existing_indices).copy()

        if rows_to_add:
            imputed_df = pd.DataFrame(rows_to_add)

            drop_helper_cols = [c for c in ["_load_is_null", "_imputed_rank"] if c in imputed_df.columns]
            if drop_helper_cols:
                imputed_df = imputed_df.drop(columns=drop_helper_cols)

            group_clean = pd.concat([group_clean, imputed_df], ignore_index=True)

        total_rows_added += len(rows_to_add)
        total_bad_rows_replaced += len(bad_existing_indices)

        print(
            f"[impute][{iso}] Zone '{name}': "
            f"found {missing_count} missing/null hourly slots, "
            f"filled {filled_count}, "
            f"replaced {len(bad_existing_indices)} null-load existing rows, "
            f"skipped {skipped_count} because no usable prior-week source existed."
        )

        all_parts.append(group_clean)

    final_df = pd.concat(all_parts, ignore_index=True)

    helper_cols = [c for c in ["_load_is_null", "_imputed_rank"] if c in final_df.columns]
    if helper_cols:
        final_df = final_df.drop(columns=helper_cols)

    final_df["Imputed"] = final_df["Imputed"].fillna("NO").astype(str)
    final_df["_imputed_rank"] = final_df["Imputed"].map({"NO": 0, "YES": 1}).fillna(0)

    before_dedup = len(final_df)
    final_df = (
        final_df.sort_values(["Name", "Hour Start", "_imputed_rank"])
        .drop_duplicates(subset=["Name", "Hour Start"], keep="last")
        .drop(columns="_imputed_rank")
        .reset_index(drop=True)
    )
    dedup_removed = before_dedup - len(final_df)

    print(f"[impute][{iso}] Imputation complete.")
    print(f"[impute][{iso}] Original rows: {original_rows}")
    print(f"[impute][{iso}] Total imputed rows added: {total_rows_added}")
    print(f"[impute][{iso}] Total null-load rows replaced: {total_bad_rows_replaced}")
    print(f"[impute][{iso}] Duplicate Name/Hour rows removed during final cleanup: {dedup_removed}")
    print(f"[impute][{iso}] Final row count after imputation: {len(final_df)}")

    return final_df


def impute_nyiso(load_df: pd.DataFrame):
    return impute_by_prior_week(load_df, iso="NYISO")


def impute_caiso(load_df: pd.DataFrame):
    return impute_by_prior_week(load_df, iso="CAISO")


# =========================
# Weather standardization
# =========================

def standardize_weather(weather_df: pd.DataFrame, iso: str) -> pd.DataFrame:
    iso = iso.upper()
    print(f"[weather] Standardizing weather data for ISO={iso}.")
    df = weather_df.copy()

    if "Hour Start" not in df.columns:
        raise ValueError("Weather file must contain 'Hour Start'.")

    df["Hour Start"] = pd.to_datetime(df["Hour Start"], errors="coerce")
    bad_ts = df["Hour Start"].isna().sum()
    if bad_ts:
        print(f"[weather] Found {bad_ts} weather rows with invalid Hour Start.")

    if iso == "NYISO":
        if "Zone" in df.columns and "Name" not in df.columns:
            df = df.rename(columns={"Zone": "Name"})
            print("[weather] Renamed weather column 'Zone' to 'Name' for NYISO.")
        if "Name" not in df.columns:
            raise ValueError("NYISO weather file must contain 'Zone' or 'Name'.")
        print(f"[weather] NYISO weather standardization complete. Rows: {len(df)}.")
        return df

    if iso == "CAISO":
        zone_col = "Zone" if "Zone" in df.columns else "Name" if "Name" in df.columns else None
        if zone_col is None:
            raise ValueError("CAISO weather file must contain 'Zone' or 'Name'.")

        sp15_rows = df[df[zone_col] == "SP15"].copy()
        sp15_rows[zone_col] = "SDGE"
        added = len(sp15_rows)
        df = pd.concat([df, sp15_rows], ignore_index=True)
        print(f"[weather] Duplicated {added} SP15 weather rows for SDGE.")

        # hard code CAISO weather mapping
        name_mapping = {
            "NP15": "PGE",
            "SP15": "SCE",
            "ZP26": "VEA",
        }
        df[zone_col] = df[zone_col].replace(name_mapping)
        print(f"[weather] Applied CAISO zone-to-utility mapping: {name_mapping}")

        if zone_col != "Name":
            df = df.rename(columns={zone_col: "Name"})
            print(f"[weather] Renamed weather column '{zone_col}' to 'Name'.")

        print(f"[weather] CAISO weather standardization complete. Rows: {len(df)}.")
        return df

    raise ValueError("iso must be either 'CAISO' or 'NYISO'.")


# =========================
# Merge + wide output
# =========================

def merge_load_and_weather(load_df: pd.DataFrame, weather_df: pd.DataFrame) -> pd.DataFrame:
    print("[merge] Merging load and weather on ['Name', 'Hour Start'] using inner join.")
    merged = pd.merge(load_df, weather_df, on=["Name", "Hour Start"], how="inner")
    print(f"[merge] Merge complete. Output rows: {len(merged)}.")
    return merged.sort_values(["Name", "Hour Start"]).reset_index(drop=True)


# # =========================
# # Visuals
# # =========================

def make_power_weather_plot(
    merged_df: pd.DataFrame,
    iso: str,
    temp_col: str = "temperature_2m",
    save_path: Optional[str | Path] = None,
):
    print(f"[plot] Building power-weather plot for ISO={iso}, temp_col='{temp_col}'.")
    df = merged_df.copy()
    df["Hour Start"] = pd.to_datetime(df["Hour Start"], errors="coerce")

    names = [n for n in sorted(df["Name"].dropna().unique()) if isinstance(n, str)]
    if not names:
        raise ValueError("No utility names available to plot.")

    print(f"[plot] Utilities to plot: {names}")
    num_utilities = len(names)

    fig, axes = plt.subplots(
        nrows=num_utilities,
        ncols=1,
        figsize=(15, 5 * num_utilities),
        sharex=True,
    )

    if num_utilities == 1:
        axes = [axes]

    for i, utility in enumerate(names):
        ax1 = axes[i]
        plot_df = df[df["Name"] == utility].sort_values("Hour Start")

        if plot_df.empty:
            print(f"[plot] Warning: No data for utility '{utility}'. Skipping.")
            continue

        if temp_col not in plot_df.columns:
            raise ValueError(f"Temperature column '{temp_col}' not found for utility '{utility}'.")

        print(f"[plot] Plotting utility '{utility}' with {len(plot_df)} rows.")

        ax1.set_ylabel("Hourly Avg Load")
        ax1.plot(
            plot_df["Hour Start"],
            plot_df["Hourly Avg Load"],
            label="Hourly Avg Load",
            alpha=0.8,
            color="#943018",
        )
        ax1.grid(True)

        ax2 = ax1.twinx()
        ax2.set_ylabel(temp_col)
        ax2.plot(
            plot_df["Hour Start"],
            plot_df[temp_col],
            label=temp_col,
            alpha=0.8,
            color="#185494",
        )

        ax1.set_title(f"{iso.upper()}: Load and {temp_col} for {utility}")

        handles1, labels1 = ax1.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(handles1 + handles2, labels1 + labels2, loc="upper left")

    axes[-1].set_xlabel("Time")
    fig.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches="tight")
        print(f"[plot] Saved plot to: {save_path}")

    print("[plot] Plot generation complete.")
    return fig


# =========================
# End-to-end pipeline
# =========================

def _list_csv_files(path_or_paths: str | Path | Sequence[str | Path]) -> list[Path]:
    """
    Accept either:
    - a single CSV file
    - a single ZIP file
    - a directory containing CSV or ZIP files
    - a sequence of CSV / ZIP file paths

    Returns a sorted list of input files.
    """
    valid_suffixes = {".csv", ".zip"}

    if isinstance(path_or_paths, (str, Path)):
        p = Path(path_or_paths)
        if p.is_dir():
            files = sorted([f for f in p.iterdir() if f.suffix.lower() in valid_suffixes])
            if not files:
                raise ValueError(f"No CSV or ZIP files found in directory: {p}")
            # print(f"[files] Found {len(files)} input files in directory: {p}")
            for f in files:
                print(f"[files]   - {f}")
            return files
        if p.is_file() and p.suffix.lower() in valid_suffixes:
            print(f"[files] Using single input file: {p}")
            return [p]
        raise ValueError(f"Path does not exist or is not a CSV/ZIP file: {p}")

    files = [Path(p) for p in path_or_paths]

    print(f"[files] Using {len(files)} explicitly provided input files.")
    for f in sorted(files):
        print(f"[files]   - {f}")
    return sorted(files)


def read_and_concat_csvs(
    path_or_paths: str | Path | Sequence[str | Path],
    add_source_file: bool = False,
):
    """
    Read and concatenate CSVs from plain CSV files or ZIP files.

    For ZIP files, every CSV inside the archive is read and appended.
    This supports NYISO folders that contain daily zipped CSV files.
    """
    import zipfile

    files = _list_csv_files(path_or_paths)
    dfs = []

    # print("[read] Reading and concatenating input files.")
    for f in files:
        suffix = f.suffix.lower()

        if suffix == ".csv":
            # print(f"[read] Reading CSV: {f}")
            df = pd.read_csv(f)
            # print(f"[read]   Rows read: {len(df)}")
            if add_source_file and "source_file" not in df.columns:
                df["source_file"] = f.name
                print(f"[read]   Added source_file='{f.name}'")
            dfs.append(df)
            continue

        if suffix == ".zip":
            # print(f"[read] Reading ZIP archive: {f}")
            with zipfile.ZipFile(f, "r") as zf:
                members = sorted([name for name in zf.namelist() if name.lower().endswith(".csv")])
                if not members:
                    raise ValueError(f"ZIP file contains no CSVs: {f}")
                # print(f"[read]   Found {len(members)} CSV members inside ZIP.")
                for member in members:
                    # print(f"[read]   Reading member: {member}")
                    with zf.open(member) as handle:
                        df = pd.read_csv(handle)
                    # print(f"[read]     Rows read: {len(df)}")
                    if add_source_file and "source_file" not in df.columns:
                        df["source_file"] = f"{f.name}::{member}"
                        print(f"[read]     Added source_file='{f.name}::{member}'")
                    dfs.append(df)
            continue

    if not dfs:
        raise ValueError("No readable CSV data found in the provided inputs.")

    out = pd.concat(dfs, ignore_index=True)
    print(f"[read] Concatenation complete. Total rows: {len(out)}")
    return out


def run_iso_pipeline(
    iso: str,
    load_csv_path: str | Path | Sequence[str | Path],
    weather_csv_path: str | Path | Sequence[str | Path],
    final_output_csv_path: str | Path,
    wide_output_csv_path: Optional[str | Path] = None,
    plot_output_path: Optional[str | Path] = None,
    temp_col: str = "temperature_2m",
) -> dict:
    """
    End-to-end pipeline.

    Inputs can be:
    - a single CSV file
    - a folder of monthly CSV files
    - a list of monthly CSV files

    For NYISO, this supports compiling individual monthly load files into one
    annual dataset before missing-data checks and imputation.
    """
    print("==================================================")
    print("Starting ISO load + weather pipeline")
    print("==================================================")

    iso = iso.upper()
    if iso not in {"CAISO", "NYISO"}:
        raise ValueError("iso must be either 'CAISO' or 'NYISO'.")

    print(f"[pipeline] ISO selected: {iso}")
    final_output_csv_path = Path(final_output_csv_path)

    add_source_file = iso == "CAISO"

    print("[pipeline] Reading raw load data.")
    raw_load = _read_and_concat_csvs(load_csv_path, add_source_file=add_source_file)

    print("[pipeline] Reading raw weather data.")
    raw_weather = _read_and_concat_csvs(weather_csv_path, add_source_file=False)

    print("[pipeline] Standardizing load data.")
    load_df = _standardize_load_columns(raw_load, iso=iso)

    print("[pipeline] Standardizing weather data.")
    weather_df = standardize_weather(raw_weather, iso=iso)

    print("[pipeline] Running missing-data summary before imputation.")
    missing_before = summarize_missing_data(load_df, weather_df)

    if iso == "NYISO":
        print("[pipeline] Running NYISO imputation.")
        load_imputed = impute_nyiso(load_df)
    else:
        print("[pipeline] Running CAISO cleaning/imputation step.")
        load_imputed = impute_caiso(load_df)

    print("[pipeline] Running missing-data summary after imputation.")
    missing_after = summarize_missing_data(load_imputed, weather_df)

    print("[pipeline] Merging load and weather.")
    merged_long = merge_load_and_weather(load_imputed, weather_df)

    final_output_csv_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"[output] Writing merged long CSV to: {final_output_csv_path}")
    merged_long.to_csv(final_output_csv_path, index=False)

    if wide_output_csv_path is None:
        wide_output_csv_path = final_output_csv_path.with_name(
            final_output_csv_path.stem + "_wide.csv"
        )
    wide_output_csv_path = Path(wide_output_csv_path)
    wide_output_csv_path.parent.mkdir(parents=True, exist_ok=True)

    if plot_output_path is None:
        plot_output_path = final_output_csv_path.with_suffix(".png")

    print("[pipeline] Creating plot.")
    fig = make_power_weather_plot(
        merged_df=merged_long,
        iso=iso,
        temp_col=temp_col,
        save_path=plot_output_path,
    )

    results = {
        "iso": iso,
        "load_input_files": [str(p) for p in _list_csv_files(load_csv_path)],
        "weather_input_files": [str(p) for p in _list_csv_files(weather_csv_path)],
        "raw_load_rows": len(raw_load),
        "raw_weather_rows": len(raw_weather),
        "merged_long_rows": len(merged_long),
        "missing_before": missing_before,
        "missing_after": missing_after,
        "merged_long_output": str(final_output_csv_path),
        "wide_output": str(wide_output_csv_path),
        "plot_output": str(plot_output_path),
        "figure": fig,
    }

    print("\n--- Pipeline Results ---")
    print(f"ISO: {results['iso']}")
    print(f"Raw Load Rows: {results['raw_load_rows']}")
    print(f"Raw Weather Rows: {results['raw_weather_rows']}")
    print(f"Merged Long Rows: {results['merged_long_rows']}")

    print("\nMissing Data Summary (Before Imputation):")
    for key, value in results["missing_before"].items():
        print(f"  {key}: {value}")

    print("\nMissing Data Summary (After Imputation):")
    for key, value in results["missing_after"].items():
        print(f"  {key}: {value}")

    print(f"\nOutput Files:")
    print(f"  Merged Long CSV: {results['merged_long_output']}")
    print(f"  Plot Image: {results['plot_output']}")

    print("\n[pipeline] Pipeline complete.")
    print("==================================================")

    return results


if __name__ == "__main__":
    pass

##Forecasting

End-to-end forecasting pipeline for a single load zone using LightGBM.

### Steps:
1. Filter and preprocess the zone data
2. Build target and feature matrix
3. Create train/test split
4. Fit recursive sktime + LGBM model
5. Generate predictions
6. Compute hourly and daily peak metrics
7. Plot predictions

In [ ]:
!pip install sktime tensorflow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import necessary packages for predictions
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

import warnings
warnings.filterwarnings("ignore")

from sktime.utils.plotting import plot_series, plot_windows
from sktime.datasets import load_airline
from sktime.forecasting.model_selection import (
    temporal_train_test_split,
    ForecastingGridSearchCV,
    ExpandingWindowSplitter,
)
from sktime.performance_metrics.forecasting import MeanSquaredError
from sktime.forecasting.compose import make_reduction, ForecastingPipeline
from sktime.transformations.series.lag import Lag
from sktime.forecasting.base import ForecastingHorizon
from sktime.datatypes._panel._convert import from_2d_array_to_nested

from lightgbm import LGBMRegressor

from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_squared_error

In [ ]:
# Read data CAISO
caiso2019 = pd.read_csv(r'/content/drive/MyDrive/Ohmies Final Project/CAISO Data/Compiled Data/caiso_2019_merged.csv')
caiso2020 = pd.read_csv(r'/content/drive/MyDrive/Ohmies Final Project/CAISO Data/Compiled Data/caiso_2020_merged.csv')
caiso2021 = pd.read_csv(r'/content/drive/MyDrive/Ohmies Final Project/CAISO Data/Compiled Data/caiso_2021_merged.csv')
caiso2022 = pd.read_csv(r'/content/drive/MyDrive/Ohmies Final Project/CAISO Data/Compiled Data/caiso_2022_merged.csv')
caiso2023 = pd.read_csv(r'/content/drive/MyDrive/Ohmies Final Project/CAISO Data/Compiled Data/caiso_2023_merged.csv')
caiso2024 = pd.read_csv(r'/content/drive/MyDrive/Ohmies Final Project/CAISO Data/Compiled Data/caiso_2024_merged.csv')
caiso2025 = pd.read_csv(r'/content/drive/MyDrive/Ohmies Final Project/CAISO Data/Compiled Data/caiso_2025_merged.csv')

In [ ]:
# Read data NYISO
nyiso_2019 = pd.read_csv('/content/drive/MyDrive/Ohmies Final Project/NYISO Data/Compiled Data/power and weather data/nyiso_2019_merged.csv')
nyiso_2020 = pd.read_csv('/content/drive/MyDrive/Ohmies Final Project/NYISO Data/Compiled Data/power and weather data/nyiso_2020_merged.csv')
nyiso_2021 = pd.read_csv('/content/drive/MyDrive/Ohmies Final Project/NYISO Data/Compiled Data/power and weather data/nyiso_2021_merged.csv')
nyiso_2022 = pd.read_csv('/content/drive/MyDrive/Ohmies Final Project/NYISO Data/Compiled Data/power and weather data/nyiso_2022_merged.csv')
nyiso_2023 = pd.read_csv('/content/drive/MyDrive/Ohmies Final Project/NYISO Data/Compiled Data/power and weather data/nyiso_2023_merged.csv')
nyiso_2024 = pd.read_csv('/content/drive/MyDrive/Ohmies Final Project/NYISO Data/Compiled Data/power and weather data/nyiso_2024_merged.csv')
nyiso_2025 = pd.read_csv('/content/drive/MyDrive/Ohmies Final Project/NYISO Data/Compiled Data/power and weather data/nyiso_2025_merged.csv')

In [ ]:
def build_sktime_LGBM_model(df, zone_name, start_date, train_days=365):

    # 1. Filter and preprocess the zone data

    # Filter for the specific zone
    zone_data = df[df['Name'] == zone_name].copy()
    zone_data['Hour Start'] = pd.to_datetime(zone_data['Hour Start'])
    zone_data = zone_data.sort_values('Hour Start').set_index('Hour Start')

    # Set frequency and handle missing values
    zone_data = zone_data.asfreq('H')
    if zone_data.isnull().values.any():
        zone_data = zone_data.interpolate(method='linear')

    # 2. Define target and feature matrix

    # Define Target (y)
    target_col = 'Hourly Avg Load'
    y = zone_data[target_col]

    # Build Feature Matrix (X)
    X = pd.DataFrame(index=zone_data.index)

    X['hr_sin'] = np.sin(2 * np.pi * zone_data.index.hour / 24)
    X['hr_cos'] = np.cos(2 * np.pi * zone_data.index.hour / 24)

    X['dow_sin'] = np.sin(2 * np.pi * zone_data.index.dayofweek / 7)
    X['dow_cos'] = np.cos(2 * np.pi * zone_data.index.dayofweek / 7)

    X['doy_sin'] = np.sin(2 * np.pi * zone_data.index.dayofyear / 365.25)
    X['doy_cos'] = np.cos(2 * np.pi * zone_data.index.dayofyear / 365.25)

    weather_cols = ['temperature_2m',
                    'relativehumidity_2m',
                    'precipitation',
                    'windspeed_10m',
                    'dewpoint_2m'
                    ]

    for col in weather_cols:
        if col in zone_data.columns:
            X[col] = zone_data[col]

    # 3. Create train/test split

    # Define windows
    test_start = pd.to_datetime(start_date)
    test_end = test_start + pd.Timedelta(hours=167)
    train_start = test_start - pd.Timedelta(days=train_days)
    train_end = test_start - pd.Timedelta(hours=1)

    # Train/test split
    y_train = y.loc[train_start : train_end]
    X_train = X.loc[train_start : train_end]
    y_test = y.loc[test_start : test_end]
    X_test = X.loc[test_start : test_end]

    # 4. Fit recursive sktime + LGBM model

    # Define Forecasting Horizon
    fh = ForecastingHorizon(y_test.index, is_relative=False)

    # Forecasting Pipeline
    forecaster = ForecastingPipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", make_reduction(
                LGBMRegressor(),
                strategy="recursive",
                window_length=168
            ))
        ]
    )

    # 5. Generate predictions
    forecaster.fit(y_train, X=X_train)
    y_pred = forecaster.predict(fh, X=X_test)

    # Metrics Calculation
    hourly_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    hourly_mape = (abs((y_test - y_pred) / y_test)).mean() * 100

    # 6. Compute hourly and daily peak metrics
    true_daily_peaks = y_test.resample("D").max()
    pred_daily_peaks = y_pred.resample("D").max()
    daily_peak_rmse = np.sqrt(mean_squared_error(true_daily_peaks, pred_daily_peaks))
    daily_peak_mape = (abs((true_daily_peaks - pred_daily_peaks) / true_daily_peaks)).mean() * 100

    # Summary Printout
    print(f"--- Results for {zone_name} ---")
    print(f"Hourly RMSE: {hourly_rmse:.2f}")
    print(f"Hourly MAPE: {hourly_mape:.2f}%")
    print(f"Daily Peak RMSE: {daily_peak_rmse:.2f}")
    print(f"Daily Peak MAPE: {daily_peak_mape:.2f}%")

    return {
        "y_train": y_train,
        "y_test": y_test,
        "y_pred": y_pred,
        "hourly_rmse": hourly_rmse,
        "hourly_mape": hourly_mape,
        "daily_peak_comparison": pd.DataFrame({"true": true_daily_peaks, "pred": pred_daily_peaks})
    }

In [ ]:
def save_sktime_results_to_csv(results, zone_name, start_date, output_path=None):

    # Save hourly forecasting results to a CSV file, assumes outputs of build_sktime_LGBM_model stored in a dictionary called results

    y_test = results["y_test"]
    y_pred = results["y_pred"]

    # Build results table
    results_df = pd.DataFrame({
        "Hour Start": y_test.index,
        "actual_load": y_test.values,
        "predicted_load": y_pred.values
    })

    results_df["error"] = results_df["actual_load"] - results_df["predicted_load"]
    results_df["abs_error"] = results_df["error"].abs()
    results_df["percent_error"] = np.where(
        results_df["actual_load"] != 0,
        results_df["abs_error"] / results_df["actual_load"] * 100,
        np.nan
    )

    # Default filename if none provided
    if output_path is None:
        safe_zone = zone_name.replace(" ", "_")
        safe_date = pd.to_datetime(start_date).strftime("%Y-%m-%d")
        output_path = f"{safe_zone}_LGBM_forecast_results_{safe_date}.csv"

    results_df.to_csv(output_path, index=False)
    print(f"Results saved to: {output_path}")

    return results_df